# Solar Yield Prediction Model Training
**Dataset:** Solar Power Generation Data  
**Link:** https://www.kaggle.com/datasets/anikannal/solar-power-generation-data  
**Output:** `solar_model.pkl`  
**Model:** XGBoost Regressor  
**Input features:** `solar_irradiance`, `temperature_avg`, `cloud_cover`, `peak_sun_hours`, `temp_delta`  
**Target:** `capacity_factor` (actual output / theoretical max, 0–1)

In [ ]:
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Load Dataset

In [ ]:
BASE = '/kaggle/input/datasets/anikannal/solar-power-generation-data'

def load_csv(path):
    df = pd.read_csv(path)
    df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'], dayfirst=True)
    return df

p1_gen     = load_csv(f'{BASE}/Plant_1_Generation_Data.csv')
p2_gen     = load_csv(f'{BASE}/Plant_2_Generation_Data.csv')
p1_weather = load_csv(f'{BASE}/Plant_1_Weather_Sensor_Data.csv')
p2_weather = load_csv(f'{BASE}/Plant_2_Weather_Sensor_Data.csv')

gen     = pd.concat([p1_gen,     p2_gen],     ignore_index=True)
weather = pd.concat([p1_weather, p2_weather], ignore_index=True)

print('Generation shape:', gen.shape)
print('Weather shape:',    weather.shape)
print('Plants:', gen['PLANT_ID'].unique())

## 2. Compute Installed Capacity Per Plant

In [ ]:
# Total plant AC output per timestamp = sum across all inverters
plant_total_ac = gen.groupby(['PLANT_ID', 'DATE_TIME'])['AC_POWER'].sum()

# Installed capacity = 95th percentile of total plant output (robust to spikes)
plant_installed_kw = plant_total_ac.groupby('PLANT_ID').quantile(0.95) / 1000
print('Plant installed capacity (kW):')
print(plant_installed_kw)

## 3. Aggregate Generation to Plant-Day Level

In [ ]:
gen['date'] = gen['DATE_TIME'].dt.date

# Sum all inverters per plant per day → convert to kWh
# watts × 0.25h (15-min slot) / 1000 = kWh
daily_gen = gen.groupby(['PLANT_ID', 'date']).agg(
    actual_kwh=('AC_POWER', lambda x: x.sum() * 0.25 / 1000)
).reset_index()

daily_gen['plant_capacity_kw'] = daily_gen['PLANT_ID'].map(plant_installed_kw)

# capacity_factor = actual_kwh / (capacity_kw × 24h)
daily_gen['capacity_factor'] = (
    daily_gen['actual_kwh'] / (daily_gen['plant_capacity_kw'] * 24)
).clip(0, 1)

print(f'Rows: {len(daily_gen)}')
print(f'Plants: {daily_gen["PLANT_ID"].nunique()}')
print(f'Unique dates: {daily_gen["date"].nunique()}')
print(f'\nCapacity factor distribution:')
print(daily_gen['capacity_factor'].describe())

## 4. Aggregate Weather to Daily

In [ ]:
weather['date'] = weather['DATE_TIME'].dt.date

daily_weather = weather.groupby(['PLANT_ID', 'date']).agg(
    solar_irradiance=('IRRADIATION',        'mean'),
    temperature_avg =('AMBIENT_TEMPERATURE', 'mean'),
    module_temp     =('MODULE_TEMPERATURE',  'mean'),
).reset_index()

df = pd.merge(daily_gen, daily_weather, on=['PLANT_ID', 'date'])
print('Merged shape:', df.shape)
df.head()

## 5. Feature Engineering

In [ ]:
# peak_sun_hours: count 15-min slots per day where IRRADIATION > 0.2, divide by 4
weather['date'] = weather['DATE_TIME'].dt.date
psh = (
    weather[weather['IRRADIATION'] > 0.2]
    .groupby(['PLANT_ID', 'date'])
    .size()
    .reset_index(name='peak_slots')
)
psh['peak_sun_hours'] = psh['peak_slots'] / 4
df = pd.merge(df, psh[['PLANT_ID', 'date', 'peak_sun_hours']],
              on=['PLANT_ID', 'date'], how='left')
df['peak_sun_hours'] = df['peak_sun_hours'].fillna(0).clip(0, 12)

# cloud_cover: irradiance drop from per-plant clear-sky baseline
plant_max_irr = df.groupby('PLANT_ID')['solar_irradiance'].transform('quantile', 0.99)
df['cloud_cover'] = (1 - df['solar_irradiance'] / plant_max_irr.clip(lower=0.001)).clip(0, 1) * 100

# temp_delta: module heats above ambient on clear days — strong clear-sky signal
df['temp_delta'] = (df['module_temp'] - df['temperature_avg']).clip(0, 40)

print(df[['solar_irradiance', 'temperature_avg', 'cloud_cover',
          'peak_sun_hours', 'temp_delta', 'capacity_factor']].describe())

In [ ]:
# Correlation check
FEATURES = ['solar_irradiance', 'temperature_avg', 'cloud_cover',
            'peak_sun_hours', 'temp_delta']
TARGET = 'capacity_factor'
print('Correlations with capacity_factor:')
print(df[FEATURES + [TARGET]].corr()[TARGET].sort_values(ascending=False))

## 6. Train Model

In [ ]:
df_clean = df[FEATURES + [TARGET]].dropna()
df_clean = df_clean[df_clean[TARGET] > 0]

X = df_clean[FEATURES].values
y = df_clean[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Target range: {y.min():.4f} – {y.max():.4f}')
print(f'Mean capacity factor: {y.mean():.4f}')

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

y_pred = model.predict(X_test)
print(f'\nMAE: {mean_absolute_error(y_test, y_pred):.4f} capacity factor')
print(f'R²:  {r2_score(y_test, y_pred):.4f}')

In [ ]:
print('Feature importances:')
for feat, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    print(f'  {feat}: {imp:.4f}')

## 7. Save Model

In [ ]:
joblib.dump(model, 'solar_model.pkl')
print('Saved: solar_model.pkl')

# Verify — feature order must match services/solar.py:
# solar_irradiance, temperature_avg, cloud_cover, peak_sun_hours, temp_delta
loaded = joblib.load('solar_model.pkl')

test_cases = [
    [0.45, 28.0, 15.0, 5.5, 12.0],  # good solar conditions
    [0.10, 32.0, 80.0, 1.0,  2.0],  # heavy cloud cover
    [0.60, 25.0,  5.0, 7.0, 18.0],  # excellent clear sky
]
for tc in test_cases:
    cf = float(np.clip(loaded.predict([tc])[0], 0, 1))
    print(f'irr={tc[0]}, temp={tc[1]}, cloud={tc[2]}%, psh={tc[3]}, delta={tc[4]} '
          f'→ capacity_factor={cf:.4f} ({cf*100:.1f}% of max)')